# Notebook 07 — First Asymmetry: Prime-Filtered Lane Dynamics

**Repo:** `mod30-residue-lanes`  
**Layer:** `notebooks/`

Notebook 01 established the symmetric baseline: complete modulo-30 cycles give equal admissible-lane counts.

Notebook 07 introduces the first asymmetry by adding prime filtering.

Constraint view:

> Prime filtering breaks perfect admissible-lane symmetry and reveals lane-local structure beyond the modulo-30 substrate.


## Goals

1. Detect repo root and create standard output directories.
2. Load MRL foundations from Notebook 00 when available.
3. Generate full admissible-lane baseline counts.
4. Filter to primes and compare lane `07` against all admissible lanes.
5. Measure lane-level prime counts, density, spacing, and window vectors.
6. Export CSV, JSON, figures, and Markdown report.
7. Create a Colab-downloadable output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "src" / "mod30_residue_lanes").exists() or (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [NOTEBOOKS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Load Notebook 00 foundation output

Preferred input:

```text
results/notebook00_mrl_foundations.json
```

If unavailable, this notebook uses standard MRL defaults.


In [ ]:
foundation_path = RESULTS_DIR / "notebook00_mrl_foundations.json"

if foundation_path.exists():
    foundation = json.loads(foundation_path.read_text())
    MODULUS = foundation["modulus"]
    ADMISSIBLE_RESIDUES = foundation["admissible_residues"]
    print("Loaded foundation:", foundation_path)
else:
    MODULUS = 30
    ADMISSIBLE_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
    print("Notebook 00 foundation not found; using defaults.")

TARGET_LANE = 7
LANE_LABEL = f"{TARGET_LANE:02d}"

print("MODULUS:", MODULUS)
print("ADMISSIBLE_RESIDUES:", ADMISSIBLE_RESIDUES)
print("TARGET_LANE:", LANE_LABEL)


## Prime helper

This notebook uses a deterministic sieve of Eratosthenes, avoiding external dependencies.


In [ ]:
def prime_sieve(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.zeros(n_max + 1, dtype=bool)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    limit = int(np.sqrt(n_max)) + 1
    for p in range(2, limit):
        if sieve[p]:
            sieve[p*p:n_max+1:p] = False

    return sieve


## Generate interval sample

Default interval: integers `1..30000`, giving 1000 complete modulo-30 cycles.

Notebook 07 uses a larger interval than Notebook 01 because prime filtering is sparse.


In [ ]:
N_MAX = 30000
values = np.arange(1, N_MAX + 1)
is_prime_array = prime_sieve(N_MAX)

df = pd.DataFrame({
    "n": values,
    "residue": values % MODULUS,
})

df["is_admissible"] = df["residue"].isin(ADMISSIBLE_RESIDUES)
df["is_prime"] = is_prime_array[values]
df["is_target_lane"] = df["residue"] == TARGET_LANE
df["cycle"] = (df["n"] - 1) // MODULUS

admissible_df = df[df["is_admissible"]].copy()
prime_df = df[df["is_prime"]].copy()
admissible_prime_df = df[df["is_admissible"] & df["is_prime"]].copy()
lane07_prime_df = df[df["is_target_lane"] & df["is_prime"]].copy()

print("Total values:", len(df))
print("Admissible values:", len(admissible_df))
print("Prime values:", len(prime_df))
print("Admissible prime values:", len(admissible_prime_df))
print("Lane 07 prime values:", len(lane07_prime_df))

lane07_prime_df.head()


## Baseline vs prime-filtered lane counts

The full admissible baseline remains symmetric. Prime filtering introduces asymmetry.


In [ ]:
baseline_counts = (
    admissible_df
    .groupby("residue")
    .size()
    .reindex(ADMISSIBLE_RESIDUES)
    .rename("baseline_count")
    .reset_index()
)

prime_counts = (
    admissible_prime_df
    .groupby("residue")
    .size()
    .reindex(ADMISSIBLE_RESIDUES)
    .rename("prime_count")
    .reset_index()
)

lane_counts = baseline_counts.merge(prime_counts, on="residue")
lane_counts["lane_label"] = lane_counts["residue"].map(lambda x: f"{x:02d}")
lane_counts["prime_density_within_lane"] = lane_counts["prime_count"] / lane_counts["baseline_count"]
lane_counts["prime_share_among_admissible_primes"] = lane_counts["prime_count"] / lane_counts["prime_count"].sum()
lane_counts["prime_count_delta_from_mean"] = lane_counts["prime_count"] - lane_counts["prime_count"].mean()
lane_counts["is_target_lane"] = lane_counts["residue"] == TARGET_LANE

lane_counts_csv = RESULTS_DIR / "notebook07_prime_filtered_lane_counts.csv"
lane_counts.to_csv(lane_counts_csv, index=False)

print(lane_counts_csv)
lane_counts


## Lane 07 prime spacing

Lane `07` no longer repeats at a fixed spacing once prime filtering is applied.


In [ ]:
lane07_primes = lane07_prime_df["n"].to_numpy()
lane07_spacing = np.diff(lane07_primes)

spacing_df = pd.DataFrame({
    "from_prime": lane07_primes[:-1],
    "to_prime": lane07_primes[1:],
    "spacing": lane07_spacing,
})

spacing_csv = RESULTS_DIR / "notebook07_lane07_prime_spacing.csv"
spacing_df.to_csv(spacing_csv, index=False)

spacing_summary = spacing_df["spacing"].describe().to_frame().T
spacing_summary["unique_spacing_count"] = int(spacing_df["spacing"].nunique())
spacing_summary["min_spacing"] = int(spacing_df["spacing"].min())
spacing_summary["max_spacing"] = int(spacing_df["spacing"].max())

print(spacing_csv)
spacing_summary


## Windowed prime lane vectors

Each window becomes an eight-entry prime-count vector ordered by:

```text
01, 07, 11, 13, 17, 19, 23, 29
```


In [ ]:
WINDOW_SIZE = 3000

window_rows = []

for start in range(1, N_MAX + 1, WINDOW_SIZE):
    stop = min(start + WINDOW_SIZE - 1, N_MAX)
    window = df[(df["n"] >= start) & (df["n"] <= stop)]
    prime_window = window[window["is_prime"] & window["is_admissible"]]
    counts = prime_window.groupby("residue").size().reindex(ADMISSIBLE_RESIDUES, fill_value=0)

    row = {
        "window_start": start,
        "window_stop": stop,
        "window_size": len(window),
        "prime_count": int(prime_window.shape[0]),
    }
    for residue, count in counts.items():
        row[f"lane_{residue:02d}"] = int(count)
    row["lane_07_share"] = int(counts.loc[TARGET_LANE]) / max(1, int(counts.sum()))
    window_rows.append(row)

window_vectors = pd.DataFrame(window_rows)

window_vectors_csv = RESULTS_DIR / "notebook07_prime_window_lane_vectors.csv"
window_vectors.to_csv(window_vectors_csv, index=False)

print(window_vectors_csv)
window_vectors.head()


## Similarity of lane 07 to all lanes

Similarity is computed from windowed prime-count trajectories.


In [ ]:
def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

lane_cols = [f"lane_{r:02d}" for r in ADMISSIBLE_RESIDUES]
target_vector = window_vectors["lane_07"].to_numpy()

similarity_rows = []
for r in ADMISSIBLE_RESIDUES:
    col = f"lane_{r:02d}"
    sim = cosine_similarity(target_vector, window_vectors[col].to_numpy())
    similarity_rows.append({
        "target_lane": "07",
        "comparison_lane": f"{r:02d}",
        "cosine_similarity": sim,
    })

similarity_df = pd.DataFrame(similarity_rows)

similarity_csv = RESULTS_DIR / "notebook07_lane07_prime_similarity.csv"
similarity_df.to_csv(similarity_csv, index=False)

print(similarity_csv)
similarity_df


## Asymmetry summary

Prime filtering turns equal admissible-lane capacity into unequal observed prime occupancy.


In [ ]:
prime_mean = lane_counts["prime_count"].mean()
prime_std = lane_counts["prime_count"].std(ddof=0)
asymmetry_score = float(prime_std / prime_mean) if prime_mean else 0.0

summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "07_lane_residue_7",
    "title": "First Asymmetry: Prime-Filtered Lane Dynamics",
    "modulus": MODULUS,
    "target_lane": TARGET_LANE,
    "target_lane_label": LANE_LABEL,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "n_max": N_MAX,
    "window_size": WINDOW_SIZE,
    "total_values": int(len(df)),
    "admissible_values": int(len(admissible_df)),
    "prime_values": int(len(prime_df)),
    "admissible_prime_values": int(len(admissible_prime_df)),
    "target_lane_prime_values": int(len(lane07_prime_df)),
    "target_lane_prime_density_within_lane": float(
        lane_counts.loc[lane_counts["residue"] == TARGET_LANE, "prime_density_within_lane"].iloc[0]
    ),
    "prime_count_mean_across_lanes": float(prime_mean),
    "prime_count_std_across_lanes": float(prime_std),
    "asymmetry_score_std_over_mean": asymmetry_score,
    "unique_lane07_prime_spacing_count": int(spacing_df["spacing"].nunique()),
    "mean_lane07_prime_spacing": float(spacing_df["spacing"].mean()),
    "constraint_view": "Prime filtering breaks perfect admissible-lane symmetry and reveals lane-local structure beyond the modulo-30 substrate.",
}

json_path = RESULTS_DIR / "notebook07_prime_filtered_lane_dynamics_summary.json"
json_path.write_text(json.dumps(summary, indent=2))

print(json_path)
print(json.dumps(summary, indent=2))


## Figures

In [ ]:
figure_names = []

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_names.append(name)
    print(path)

# 1. Baseline vs prime-filtered lane counts
x = np.arange(len(lane_counts))
width = 0.38

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, lane_counts["baseline_count"], width, label="admissible baseline")
plt.bar(x + width/2, lane_counts["prime_count"], width, label="prime-filtered")
plt.xticks(x, lane_counts["lane_label"])
plt.xlabel("Residue lane")
plt.ylabel("Count")
plt.title("Notebook 07 — Baseline vs Prime-Filtered Lane Counts")
plt.legend()
savefig("notebook07_baseline_vs_prime_lane_counts.png")

# 2. Prime share by lane
plt.figure(figsize=(8, 4))
plt.bar(lane_counts["lane_label"], lane_counts["prime_share_among_admissible_primes"])
plt.xlabel("Residue lane")
plt.ylabel("Prime share")
plt.title("Notebook 07 — Prime Share Among Admissible Lanes")
savefig("notebook07_prime_share_by_lane.png")

# 3. Lane 07 prime positions across cycles
plt.figure(figsize=(10, 4))
plt.scatter(lane07_prime_df["cycle"], lane07_prime_df["n"], s=12)
plt.xlabel("Modulo-30 cycle")
plt.ylabel("Prime n")
plt.title("Notebook 07 — Lane 07 Prime Positions Across Cycles")
savefig("notebook07_lane07_prime_positions.png")

# 4. Lane 07 prime spacing distribution
plt.figure(figsize=(8, 4))
plt.hist(spacing_df["spacing"], bins=30)
plt.xlabel("Prime-to-prime spacing within lane 07")
plt.ylabel("Frequency")
plt.title("Notebook 07 — Lane 07 Prime Spacing Distribution")
savefig("notebook07_lane07_prime_spacing_distribution.png")

# 5. Windowed prime lane vectors
matrix = window_vectors[lane_cols].to_numpy()
plt.figure(figsize=(10, 5))
plt.imshow(matrix, aspect="auto")
plt.xticks(range(len(lane_cols)), [c.replace("lane_", "") for c in lane_cols])
plt.yticks(range(len(window_vectors)), window_vectors["window_start"])
plt.xlabel("Residue lane")
plt.ylabel("Window start")
plt.colorbar(label="Prime count")
plt.title("Notebook 07 — Windowed Prime Lane Vectors")
savefig("notebook07_prime_window_lane_vectors.png")

# 6. Lane 07 similarity
plt.figure(figsize=(8, 4))
plt.bar(similarity_df["comparison_lane"], similarity_df["cosine_similarity"])
plt.xlabel("Comparison lane")
plt.ylabel("Cosine similarity to lane 07")
plt.ylim(0, 1.05)
plt.title("Notebook 07 — Lane 07 Prime Similarity Across Windows")
savefig("notebook07_lane07_prime_similarity.png")

# 7. Delta from mean
plt.figure(figsize=(8, 4))
plt.bar(lane_counts["lane_label"], lane_counts["prime_count_delta_from_mean"])
plt.axhline(0, linewidth=1)
plt.xlabel("Residue lane")
plt.ylabel("Prime count delta from mean")
plt.title("Notebook 07 — Prime Count Asymmetry Across Lanes")
savefig("notebook07_prime_count_delta_from_mean.png")


## Build Markdown report

The report uses repo-relative links for CSV, JSON, and figure outputs.


In [ ]:
report_path = REPORTS_DIR / "report_07_prime_filtered_lane_dynamics.md"

output_links = "\n".join([
    '- Prime-filtered lane counts CSV: <a href="../results/notebook07_prime_filtered_lane_counts.csv">`results/notebook07_prime_filtered_lane_counts.csv`</a>',
    '- Lane 07 prime spacing CSV: <a href="../results/notebook07_lane07_prime_spacing.csv">`results/notebook07_lane07_prime_spacing.csv`</a>',
    '- Windowed prime lane vectors CSV: <a href="../results/notebook07_prime_window_lane_vectors.csv">`results/notebook07_prime_window_lane_vectors.csv`</a>',
    '- Lane 07 prime similarity CSV: <a href="../results/notebook07_lane07_prime_similarity.csv">`results/notebook07_lane07_prime_similarity.csv`</a>',
    '- Summary JSON: <a href="../results/notebook07_prime_filtered_lane_dynamics_summary.json">`results/notebook07_prime_filtered_lane_dynamics_summary.json`</a>',
] + [
    f'- Figure: <a href="../figures/{name}">`figures/{name}`</a>'
    for name in figure_names
])

report = f"""# Report 07 — First Asymmetry: Prime-Filtered Lane Dynamics

Notebook 07 introduces the first asymmetry after Notebook 01 established the symmetric modulo-30 baseline.

Constraint view:

> Prime filtering breaks perfect admissible-lane symmetry and reveals lane-local structure beyond the modulo-30 substrate.

## Generated outputs

{output_links}

## Summary

| Metric | Value |
|---|---:|
| Modulus | {MODULUS} |
| Target lane | {LANE_LABEL} |
| Total interval values | {len(df)} |
| Admissible values | {len(admissible_df)} |
| Prime values | {len(prime_df)} |
| Admissible prime values | {len(admissible_prime_df)} |
| Lane 07 prime values | {len(lane07_prime_df)} |
| Asymmetry score std/mean | {asymmetry_score:.6f} |
| Mean lane 07 prime spacing | {spacing_df["spacing"].mean():.6f} |

## Prime-filtered lane counts

{lane_counts.to_markdown(index=False)}

## Lane 07 prime spacing summary

{spacing_summary.to_markdown(index=False)}

## Lane 07 prime similarity

{similarity_df.to_markdown(index=False)}

## Interpretation

- Notebook 01 showed the symmetric baseline of complete modulo-30 admissible cycles.
- Notebook 07 adds prime filtering as the first extra constraint.
- Prime filtering breaks equal lane occupancy while retaining the same eight-lane substrate.
- Lane `07` becomes the first asymmetric-lift notebook in the MRL sequence.
- Windowed prime-count vectors are now meaningful inputs for later RML, CGCS, and cross-lane correlation analysis.

## Next step

Notebook 11 can study window drift: how prime-filtered lane vectors change when window boundaries no longer align cleanly with modulo-30 cycles.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:5000])


## Render generated figures in notebook

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook07_prime_filtered_lane_dynamics_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook07_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in FIGURES_DIR.glob("notebook07_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in REPORTS_DIR.glob("report_07_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook07_prime_filtered_lane_dynamics_outputs.zip"
# export_path = REPO_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook07_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#         for p in folder.glob("report_07_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
